In [15]:
import torch
import torch.nn as nn

In [16]:
MODEL_PATH = "compatika_lstm.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [17]:

ckpt = torch.load(MODEL_PATH, map_location=device)
itos = ckpt["itos"]
stoi = ckpt["stoi"]

PAD_IDX = stoi["<pad>"]
UNK_IDX = stoi["<unk>"]
SOS_IDX = stoi["<sos>"]
EOS_IDX = stoi["<eos>"]

EMBED_SIZE = ckpt["config"]["embed"]
HIDDEN_SIZE = ckpt["config"]["hidden"]
NUM_LAYERS = ckpt["config"]["layers"]
MAX_LEN = 60

C:\Users\aman\AppData\Local\Temp\ipykernel_9924\2307866413.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(MODEL_PATH, map_location=device)


In [18]:
# ------- Tokenizer -------
import re
def simple_tokenize(text):
    return re.findall(r"\w+|[^\s\w]", text.lower())

def encode_tokens(tokens):
    ids = [stoi.get(t, UNK_IDX) for t in tokens]
    ids.append(EOS_IDX)
    return torch.tensor(ids, dtype=torch.long)

def decode_ids(ids):
    toks = []
    for i in ids:
        if i == EOS_IDX:
            break
        if i in (PAD_IDX, SOS_IDX):
            continue
        toks.append(itos[i])
    return " ".join(toks)

In [19]:
# ------- Define model (same as before) -------
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_size, hid_size, n_layers=1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_size, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(emb_size, hid_size, num_layers=n_layers, batch_first=True)
    def forward(self, x):
        e = self.emb(x)
        outputs, (h, c) = self.lstm(e)
        return outputs, (h, c)

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_size, hid_size, n_layers=1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_size, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(emb_size, hid_size, num_layers=n_layers, batch_first=True)
        self.fc = nn.Linear(hid_size, vocab_size)
    def forward(self, inp, hidden):
        e = self.emb(inp)
        out, hidden = self.lstm(e, hidden)
        logits = self.fc(out.squeeze(1))
        return logits, hidden

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def predict(self, sentence):
        tokens = simple_tokenize(sentence)
        src = encode_tokens(tokens).unsqueeze(0).to(device)

        enc_out, hidden = self.encoder(src)

        dec_input = torch.tensor([[SOS_IDX]], device=device)
        dec_hidden = hidden

        result = []
        for _ in range(MAX_LEN):
            logits, dec_hidden = self.decoder(dec_input, dec_hidden)
            nxt = logits.argmax(dim=1)
            if nxt.item() == EOS_IDX:
                break
            result.append(nxt.item())
            dec_input = nxt.unsqueeze(1)

        return decode_ids(result)

# ------- Load model -------
vocab_size = len(itos)
enc = Encoder(vocab_size, EMBED_SIZE, HIDDEN_SIZE, NUM_LAYERS)
dec = Decoder(vocab_size, EMBED_SIZE, HIDDEN_SIZE, NUM_LAYERS)
model = Seq2Seq(enc, dec).to(device)
model.load_state_dict(ckpt["model_state"])
model.eval()



Seq2Seq(
  (encoder): Encoder(
    (emb): Embedding(146, 128, padding_idx=0)
    (lstm): LSTM(128, 256, num_layers=2, batch_first=True)
  )
  (decoder): Decoder(
    (emb): Embedding(146, 128, padding_idx=0)
    (lstm): LSTM(128, 256, num_layers=2, batch_first=True)
    (fc): Linear(in_features=256, out_features=146, bias=True)
  )
)

In [28]:
# ------- TEST ONE INPUT -------

user_input = input("USER: ")
print("\nCOMPATIKA:", model.predict(user_input))


COMPATIKA: you that way — loneliness can surprise anyone .


In [31]:


sample_user_inputs = [
    " I felt embarrassed when I was going through a stressful moment.",
    "I became grateful after hearing unexpected news.",
    "I was feeling embarrassed when studying with someone happened.",
    "I felt grateful during studying with someone.",
    "It made me feel embarrassed while remembering something."
]

for i in range(5):
    user_input = sample_user_inputs[i]
    reply = model.predict(user_input)
    print(f"USER {i+1}: {user_input}")
    print(f"COMPATIKA {i+1}: {reply}\n")


USER 1:  I felt embarrassed when I was going through a stressful moment.
COMPATIKA 1: ah , that must ’ ve been awkward — it happens to all of us .

USER 2: I became grateful after hearing unexpected news.
COMPATIKA 2: i can see why that touched you deeply .

USER 3: I was feeling embarrassed when studying with someone happened.
COMPATIKA 3: ah , that must ’ ve been awkward — it happens to all of us .

USER 4: I felt grateful during studying with someone.
COMPATIKA 4: i can see why that touched you deeply .

USER 5: It made me feel embarrassed while remembering something.
COMPATIKA 5: ah , that must ’ ve been awkward — it happens to all of us .

